# Weeks 5–8 Consolidation Practical (Solutions)

This notebook contains example solutions for `09_practical_STUDENT.ipynb`. As with previous
weeks, there is often more than one valid way to solve these problems — if your code produces
the correct result via a different route, that's fine!

---


## Section 1: `numpy.ndarray` fundamentals (Week 5)

In [ ]:
import numpy as np

### Exercise 1.1

In [ ]:
data = np.arange(1, 21, dtype=np.int32)
print(data.shape, data.dtype)

data_slice = data[:5]
data_slice[0] = -1

print(data)
# data changed because a slice of a numpy array is a VIEW onto the same
# underlying memory, not a copy (unlike slicing a python list).


### Exercise 1.2

In [ ]:
def winsorize(data, cut_off=0.1):
    """
    Winsorize a numpy.ndarray in place.

    Parameters:
    -----------
    data: numpy.ndarray
        numeric data to winsorize (modified in place)
    cut_off: float
        proportion to winsorize at each tail (default 0.1 -> 10th/90th percentiles)
    """
    low = np.percentile(data, cut_off * 100)
    high = np.percentile(data, (1 - cut_off) * 100)

    data[np.where(data < low)] = low
    data[np.where(data > high)] = high


data = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11])
winsorize(data)
print(data)


## Section 2: Statistical procedures in `numpy` (Week 6)

### Exercise 2.1

In [ ]:
SAMPLE_SIZE = 10_000

rng = np.random.default_rng(42)
samples = rng.normal(size=SAMPLE_SIZE)


def basic_descriptives(data):
    """
    Returns mean, stdev, and 1st and 99th percentile of a 1D numpy.ndarray

    Parameters:
    ------------
    data: numpy.ndarray
        numeric data to analyse

    Returns:
    --------
    (float, float, float, float)
    """
    mean = data.mean()
    std = data.std()
    per_1st = np.percentile(data, 1)
    per_99th = np.percentile(data, 99)

    return mean, std, per_1st, per_99th


results = basic_descriptives(samples)
print(results)


### Exercise 2.2

In [ ]:
def prob_great_than_or_equal_to(data, x):
    """Return the proportion of the dataset that is greater than or equal to x"""
    return (data >= x).sum() / data.shape[0]


def prob_less_than_or_equal_to(data, x):
    """Return the proportion of the dataset that is less than or equal to x"""
    return (data <= x).sum() / data.shape[0]


p_upper = prob_great_than_or_equal_to(samples, 1.96)
p_lower = prob_less_than_or_equal_to(samples, -1.96)

print('P(X >= 1.96):', p_upper)
print('P(X <= -1.96):', p_lower)
print('P(-1.96 < X < 1.96):', 1 - (p_upper + p_lower))
# This is close to the expected ~95% for a standard normal distribution.


## Section 3: `pandas.DataFrame` basics and filtering (Week 7)

In [ ]:
import pandas as pd

### Exercise 3.1

In [ ]:
unique_patient_id = ['het1m', 'ulr33f', 'ham1f', 'tru4m']
surname = ['Hetfield', 'Ulrich', 'Hammett', 'Trujillo']
age = [50, 89, 32, 65]
female = [False, True, True, False]
first_appointment = [False, True, True, True]

patient_bookings = pd.DataFrame()
patient_bookings['unique_id'] = pd.Series(unique_patient_id, dtype=str)
patient_bookings['surname'] = pd.Series(surname, dtype=str)
patient_bookings['age'] = pd.Series(age, dtype=np.uint8)
patient_bookings['female'] = pd.Series(female, dtype=bool)
patient_bookings['first_appoint'] = pd.Series(first_appointment, dtype=bool)

patient_bookings = patient_bookings.set_index('unique_id')
patient_bookings


In [ ]:
# filter for index 'ham1f'
patient_bookings.loc['ham1f']


In [ ]:
# filter for both 'tru4m' and 'ulr33f'
to_find = ['tru4m', 'ulr33f']
patient_bookings.loc[to_find]


### Exercise 3.2

In [ ]:
DATA_URL = ('https://raw.githubusercontent.com/health-data-science-OR/'
            'hpdm139-datasets/main/sw_imaging.csv')

sw_imaging = pd.read_csv(DATA_URL)
sw_imaging.info()


In [ ]:
print(sw_imaging.shape)
sw_imaging.head(2)


In [ ]:
# only Magnetic Resonance Imaging
sw_imaging[sw_imaging['imaging_type'] == 'Magnetic Resonance Imaging']


In [ ]:
# only the Royal Devon and Exeter NHS Foundation Trust
sw_imaging[sw_imaging['provider'] == 'Royal Devon and Exeter NHS Foundation Trust']


In [ ]:
# n_referrals > 100,000 AND mdn_days_rtt > 0
sw_imaging[(sw_imaging['n_referrals'] > 100_000) &
           (sw_imaging['mdn_days_rtt'] > 0)]


### Exercise 3.3

In [ ]:
results = sw_imaging.groupby(by='provider')['n_referrals'].sum()
results


In [ ]:
# save the results
results.to_csv('total_referrals.csv', index=True)


In [ ]:
# provider with the highest number of referrals
results[results == results.max()]


## Section 4: Data wrangling and `matplotlib` (Week 8)

In [ ]:
import matplotlib.pyplot as plt

### Exercise 4.1

In [ ]:
LONG_URL = ('https://raw.githubusercontent.com/health-data-science-OR/'
            'hpdm139-datasets/main/synt_ed_long.csv')

long_df = pd.read_csv(LONG_URL)
long_df.head()


In [ ]:
long_df.info()

In [ ]:
def ed_data_to_wide(filepath):
    """
    Return the ED data in wide format.

    1. Pivot table
    2. Transpose and drop the (attends, hosp_i) multi-index
    3. Rename columns 0, 1, 2, 3 to hosp1, hosp2, hosp3, hosp4
    4. Index to DateTimeIndex
    5. Convert attendance numbers from int64 to int16

    Params:
    -------
    filepath: str
        Path to long format file

    Returns:
    --------
    pandas.DataFrame
    """
    translated_names = {0: 'hosp1', 1: 'hosp2', 2: 'hosp3', 3: 'hosp4'}
    dtypes = {'hosp1': np.int16, 'hosp2': np.int16, 'hosp3': np.int16, 'hosp4': np.int16}

    df = (pd.read_csv(filepath)
          .pivot_table(values='attends', index='date', columns='hosp')
          .T.reset_index(drop=True).T
          .rename(columns=translated_names)
          .assign(date=lambda x: pd.to_datetime(x.index))
          .set_index('date')
          .astype(dtypes))

    return df


wide_df = ed_data_to_wide(LONG_URL)
wide_df.info()


In [ ]:
wide_df.head()

### Exercise 4.2

In [ ]:
fig = plt.figure(figsize=(12, 3))
ax = fig.add_subplot()

ax.plot(wide_df['hosp1'], lw=2)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Attendances', fontsize=12)
ax.grid(ls='--')
ax.tick_params(axis='both', labelsize=12)

fig.savefig('hosp1_ed.png', dpi=300)
plt.show()


---
## End of consolidation practical solutions